In [4]:
import os
import sys
import yaml
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

if os.path.basename(os.getcwd()) != 'audio_event_detection':
    os.chdir('..')
from models.ast_model import AudioSpectrogramTransformer
from scripts.train import Trainer

from scripts.evaluate import ModelEvaluator
from utils.dataset import AudioEventDataset

In [5]:
# Load tập test
config_path = 'configs/config.yaml'
metadata_df = pd.read_csv('data/processed/spectrograms/processed_metadata.csv')
train_df, temp_df = train_test_split(metadata_df, test_size=0.2, random_state=42, stratify=metadata_df['label'])

with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)
batch_size = config_dict['training']['batch_size']
num_workers = config_dict.get('hardware', {}).get('num_workers', 4)
pin_memory = config_dict.get('hardware', {}).get('pin_memory', True)

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
test_dataset = AudioEventDataset(test_df, config_path, mode='test')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                         num_workers=num_workers, pin_memory=pin_memory)


In [9]:
# 1. Kiểm tra số lượng class định nghĩa trong cấu hình (config)
print(f"Số lượng class tối đa (theo config): {test_dataset.num_classes}")

# 2. Kiểm tra các class thực tế có trong tập test_df
unique_labels = test_df['label'].unique()
print(f"Số lượng class thực tế trong tập test: {len(unique_labels)}")

# 3. Hiển thị chi tiết mapping giữa ID và tên class (nếu có cột target_class)
if 'target_class' in test_df.columns:
    class_mapping = test_df[['label', 'target_class']].drop_duplicates().sort_values('label')
    print("\nDanh sách các class trong tập test:")
    print(class_mapping.to_string(index=False))
else:
    print(f"Danh sách các Label ID xuất hiện: {sorted(unique_labels)}")

# 4. Kiểm tra nhanh qua test_loader (để đảm bảo dữ liệu đầu ra đúng)
images, labels = next(iter(test_loader))
print(f"\nKích thước một batch: {images.shape}")
print(f"Số lượng label trong một batch: {len(labels)}")


Số lượng class tối đa (theo config): 8
Số lượng class thực tế trong tập test: 8

Danh sách các class trong tập test:
 label   target_class
     0        gunshot
     1      explosion
     2          siren
     3 glass_breaking
     4         scream
     5       dog_bark
     6 fire_crackling
     7         normal

Kích thước một batch: torch.Size([32, 1, 128, 173])
Số lượng label trong một batch: 32


In [7]:
best_model_path = os.path.join(os.getcwd(), config_dict['paths']['checkpoint_dir'], 'best_model.pth')
Evaluator = ModelEvaluator(
    model_path=best_model_path
)

Evaluator initialized on cpu


In [8]:
Evaluator.evaluate(test_loader)


Evaluating model...


Evaluation: 100%|██████████| 34/34 [01:06<00:00,  1.96s/it]


{'metrics': {'accuracy': 0.8994413407821229,
  'precision': 0.6045299555465751,
  'recall': 0.6236194800346813,
  'f1_score': 0.5939722345935707,
  'precision_gunshot': np.float64(0.75),
  'recall_gunshot': np.float64(0.9473684210526315),
  'f1_gunshot': np.float64(0.8372093023255814),
  'precision_explosion': np.float64(0.2),
  'recall_explosion': np.float64(0.25),
  'f1_explosion': np.float64(0.2222222222222222),
  'precision_siren': np.float64(0.8367346938775511),
  'recall_siren': np.float64(0.8817204301075269),
  'f1_siren': np.float64(0.8586387434554974),
  'precision_glass_breaking': np.float64(1.0),
  'recall_glass_breaking': np.float64(0.5),
  'f1_glass_breaking': np.float64(0.6666666666666666),
  'precision_scream': np.float64(0.0),
  'recall_scream': np.float64(0.0),
  'f1_scream': np.float64(0.0),
  'precision_dog_bark': np.float64(0.7227722772277227),
  'recall_dog_bark': np.float64(0.73),
  'f1_dog_bark': np.float64(0.7263681592039801),
  'precision_fire_crackling': np.fl